# E1 — Phân loại ảnh: Softmax, MLP, CNN

**CO5085 · Học sâu và ứng dụng trong thị giác máy tính · HK261** — tập dữ liệu: Fashion-MNIST.

Notebook này tự chứa toàn bộ code của E1 và chạy được từ đầu đến cuối (khoảng 3 phút trên GPU T4).

| Yêu cầu của đề | Phần trong notebook |
|---|---|
| 1. Softmax classifier trên vector flatten | Mục *Mô hình* — `SoftmaxClassifier` |
| 2. MLP nhiều tầng trên vector flatten | `MLPClassifier` |
| 3. CNN (conv/pool/BN + head) | `CNNClassifier` |
| 4. Tự viết training loop, đánh giá val/test | Mục *Vòng lặp huấn luyện* |
| 5. So sánh accuracy, số tham số, đường cong, lỗi | Mục *Kết quả* và *Phân tích lỗi* |

## Thiết lập
Chạy lần lượt từ trên xuống. Trên Colab: *Runtime → Change runtime type → T4 GPU*.

In [ ]:
import csv
import json
import math
import os
import random
import time
from dataclasses import asdict, dataclass, field

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| thiết bị:",
      torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU (sẽ chậm, nên bật GPU)")

In [ ]:
# ---- hằng số và hàm vẽ/đọc kết quả (không cần PyTorch)
from matplotlib.ticker import MaxNLocator

CLASSES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
           "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
SHORT = ["T-shirt", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Boot"]
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
INK2, MUTED, GRID = "#52514e", "#8a8984", "#e4e3df"

plt.rcParams.update({
    "figure.dpi": 110, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "legend.frameon": False, "lines.linewidth": 1.8, "savefig.bbox": "tight",
})


def load_run(run_dir):
    """Đọc lại một lần chạy đã lưu: summary.json, history.csv, logits trên tập test."""
    with open(os.path.join(run_dir, "summary.json"), encoding="utf-8") as f:
        summary = json.load(f)
    run = {"summary": summary, "history": pd.read_csv(os.path.join(run_dir, "history.csv"))}
    npz = os.path.join(run_dir, "test_outputs.npz")
    if os.path.isfile(npz):
        d = np.load(npz)
        run["logits"], run["targets"] = d["logits"], d["targets"]
        run["preds"] = run["logits"].argmax(1)
    return run


def summary_row(label, run):
    s = run["summary"]
    return {"Mô hình": label, "Số tham số": s["params"],
            "Train acc (%)": round(100 * s["final_train_acc"], 2),
            "Val acc tốt nhất (%)": round(100 * s["best_val_acc"], 2), "Epoch tốt nhất": s["best_epoch"],
            "Test acc (%)": round(100 * s["test_acc"], 2), "Test loss": round(s["test_loss"], 4),
            "Giây/epoch": round(s["mean_epoch_time"], 2), "Suy luận test (s)": round(s["test_inference_time"], 3)}


def plot_curves(runs, labels, colors, styles=None, save_as=None):
    """Loss và accuracy theo epoch. Nét chấm: train; nét liền/gạch: validation."""
    styles = styles or {}
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
    for key, run in runs.items():
        h, c, ls = run["history"], colors[key], styles.get(key, "-")
        ax1.plot(h["epoch"], h["train_loss"], color=c, ls=":", lw=1.3)
        ax1.plot(h["epoch"], h["val_loss"], color=c, ls=ls)
        ax2.plot(h["epoch"], 100 * h["train_acc"], color=c, ls=":", lw=1.3)
        ax2.plot(h["epoch"], 100 * h["val_acc"], color=c, ls=ls, label=labels[key])
    ax1.set(xlabel="epoch", ylabel="cross-entropy loss", title="Loss")
    ax2.set(xlabel="epoch", ylabel="accuracy (%)", title="Accuracy")
    ax2.legend(fontsize=8)
    for ax in (ax1, ax2):
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    fig.text(0.5, -0.02, "nét chấm: train · nét liền/gạch: validation", ha="center", color=INK2)
    fig.tight_layout()
    if save_as:
        fig.savefig(save_as)
    plt.show()


def confusion(targets, preds, k=10):
    cm = np.zeros((k, k), dtype=int)
    np.add.at(cm, (targets, preds), 1)
    return cm


def softmax_np(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)

In [ ]:
EXERCISE = "E1"
SEED = 42
DATA_ROOT = "data"
RESULTS = "results/E1"
FIGURES = "figures/E1"
BATCH_SIZE = 128
QUICK = False          # True: mỗi mô hình chỉ 1 epoch, để chạy thử toàn bộ notebook thật nhanh
                       # (chạy thử xong nhớ xóa thư mục results/ trước khi chạy thật)
EPOCHS = 1 if QUICK else 15
LR = 1e-3
WEIGHT_DECAY = 1e-4
E1_NAMES = ["softmax", "mlp", "cnn"]
LABEL = {"softmax": "Softmax", "mlp": "MLP", "cnn": "CNN"}
COLOR = {"softmax": SERIES[0], "mlp": SERIES[1], "cnn": SERIES[2]}
os.makedirs(RESULTS, exist_ok=True)
os.makedirs(FIGURES, exist_ok=True)

### Seed, đo thời gian, lưu file

In [ ]:
def set_seed(seed: int = 42, deterministic: bool = True) -> None:
    """Cố định mọi nguồn ngẫu nhiên để kết quả có thể tái lập."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def count_params(model: torch.nn.Module, trainable_only: bool = True) -> int:
    """Tổng số phần tử của các tham số (p.numel())."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad or not trainable_only)


def sync(device: torch.device) -> None:
    """Chờ GPU chạy xong trước khi đọc đồng hồ, nếu không thời gian đo sẽ sai."""
    if device.type == "cuda":
        torch.cuda.synchronize()


class Timer:
    """Dùng với `with Timer(device) as t: ...` rồi đọc t.elapsed (giây)."""

    def __init__(self, device: torch.device):
        self.device = device
        self.elapsed = 0.0

    def __enter__(self):
        sync(self.device)
        self._start = time.perf_counter()
        return self

    def __exit__(self, *exc):
        sync(self.device)
        self.elapsed = time.perf_counter() - self._start
        return False


def save_json(obj, path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def load_json(path: str):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def environment_info() -> dict:
    """Ghi lại môi trường chạy để đưa vào báo cáo."""
    info = {
        "torch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    }
    try:
        import torchvision

        info["torchvision"] = torchvision.__version__
    except ImportError:
        pass
    return info

## Dữ liệu: Fashion-MNIST

- 60 000 ảnh train gốc được hoán vị bằng seed cố định: **54 000 train**, **6 000 validation**.
  10 000 ảnh test gốc chỉ dùng cho kết quả cuối.
- Ảnh đưa về $[0,1]$ rồi chuẩn hóa bằng mean/std tính **chỉ trên phần train**. Không tăng cường dữ liệu.
- Toàn bộ ảnh được giữ sẵn dưới dạng tensor trong bộ nhớ (28×28 nên rất nhẹ), mỗi epoch không phải đọc lại từ đĩa.
- Cả ba bài dùng đúng cùng split này (cùng seed), nên kết quả so sánh được với nhau.

In [ ]:
CLASSES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]
NUM_CLASSES = 10
IMAGE_SHAPE = (1, 28, 28)


@dataclass
class DataBundle:
    train: DataLoader
    val: DataLoader
    test: DataLoader
    mean: float
    std: float
    sizes: dict


def _load_raw(root: str):
    from torchvision import datasets

    train = datasets.FashionMNIST(root, train=True, download=True)
    test = datasets.FashionMNIST(root, train=False, download=True)
    # .data: uint8 (N, 28, 28), .targets: int64 (N,)
    return train.data, train.targets, test.data, test.targets


def split_indices(n_total: int, val_size: int, seed: int):
    """Trả về (train_idx, val_idx) từ một hoán vị cố định."""
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(n_total, generator=g)
    return perm[val_size:], perm[:val_size]


def get_data(root: str = "data", batch_size: int = 128, val_size: int = 6000,
             split_seed: int = 42, loader_seed: int = 0, eval_batch_size: int = 1000) -> DataBundle:
    x_tr_all, y_tr_all, x_te, y_te = _load_raw(root)
    train_idx, val_idx = split_indices(len(x_tr_all), val_size, split_seed)

    x_tr = x_tr_all[train_idx].float().div_(255.0)
    x_va = x_tr_all[val_idx].float().div_(255.0)
    x_te = x_te.float().div_(255.0)

    mean, std = x_tr.mean().item(), x_tr.std().item()

    def prep(x):  # (N, 28, 28) -> (N, 1, 28, 28), chuẩn hóa theo thống kê của tập train
        return ((x - mean) / std).unsqueeze(1)

    ds_tr = TensorDataset(prep(x_tr), y_tr_all[train_idx])
    ds_va = TensorDataset(prep(x_va), y_tr_all[val_idx])
    ds_te = TensorDataset(prep(x_te), y_te)

    # Generator riêng cho DataLoader: cùng loader_seed -> cùng thứ tự batch giữa các lần chạy.
    g = torch.Generator().manual_seed(loader_seed)
    pin = torch.cuda.is_available()
    train_loader = DataLoader(ds_tr, batch_size=batch_size, shuffle=True, generator=g, pin_memory=pin)
    val_loader = DataLoader(ds_va, batch_size=eval_batch_size, shuffle=False, pin_memory=pin)
    test_loader = DataLoader(ds_te, batch_size=eval_batch_size, shuffle=False, pin_memory=pin)

    sizes = {"train": len(ds_tr), "val": len(ds_va), "test": len(ds_te)}
    return DataBundle(train_loader, val_loader, test_loader, mean, std, sizes)


def black_value(mean: float, std: float) -> float:
    """Giá trị của một pixel đen (0) sau khi chuẩn hóa; dùng để che một phần ảnh ở E3."""
    return (0.0 - mean) / std

In [ ]:
data = get_data(DATA_ROOT, batch_size=BATCH_SIZE, loader_seed=SEED)
print("Kích thước:", data.sizes, "| mean = %.4f, std = %.4f" % (data.mean, data.std))

# Ảnh test gốc (uint8, chưa chuẩn hóa) để hiển thị
TEST_IMAGES = datasets.FashionMNIST(DATA_ROOT, train=False, download=True).data.numpy()
TEST_LABELS = datasets.FashionMNIST(DATA_ROOT, train=False, download=True).targets.numpy()

fig, axes = plt.subplots(2, 10, figsize=(12, 2.8))
for c in range(10):
    idx = np.where(TEST_LABELS == c)[0][:2]
    for r in range(2):
        axes[r, c].imshow(TEST_IMAGES[idx[r]], cmap="gray_r")
        axes[r, c].axis("off")
    axes[0, c].set_title(CLASSES[c], fontsize=8)
plt.suptitle("Hai ảnh mẫu của mỗi lớp")
plt.show()

## Vòng lặp huấn luyện tự viết

Mỗi bước đi đúng trình tự **forward → loss → `backward()` → `optimizer.step()`**, không dùng `trainer.fit()`.

- **Loss:** `nn.CrossEntropyLoss` nhận logits (đã gồm log-softmax), nên mô hình không đặt softmax ở cuối `forward`.
- **Optimizer:** AdamW, không áp weight decay cho bias, tham số chuẩn hóa, token CLS và positional embedding.
- **Learning rate:** đổi theo từng bước, warmup tuyến tính (nếu có) rồi giảm theo cosine về 0.
- **Chọn mô hình:** sau mỗi epoch đánh giá trên validation, giữ checkpoint có val accuracy cao nhất;
  tập test chỉ được đánh giá một lần bằng checkpoint đó.
- **Lưu kết quả:** mỗi lần chạy ghi `history.csv`, `summary.json`, `best.pt`, `test_outputs.npz`
  (cùng định dạng với script trong repo, nên `scripts/make_report_assets.py` dùng lại được).

In [ ]:
@dataclass
class TrainConfig:
    epochs: int = 15
    lr: float = 1e-3
    weight_decay: float = 1e-4
    warmup_epochs: float = 0.0      # số epoch tăng LR tuyến tính từ 0 trước khi giảm theo cosine
    grad_clip: float = 0.0          # 0 = không cắt gradient
    label_smoothing: float = 0.0
    seed: int = 42
    extra: dict = field(default_factory=dict)  # thông tin riêng của từng bài (kiến trúc, tokenizer...)


def make_optimizer(model: nn.Module, cfg: TrainConfig) -> torch.optim.Optimizer:
    """AdamW; không áp weight decay lên bias, LayerNorm/BatchNorm, CLS token và positional embedding."""
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim <= 1 or name.endswith(".bias") or "pos_embed" in name or "cls_token" in name:
            no_decay.append(p)
        else:
            decay.append(p)
    groups = [
        {"params": decay, "weight_decay": cfg.weight_decay},
        {"params": no_decay, "weight_decay": 0.0},
    ]
    groups = [g for g in groups if g["params"]]
    return torch.optim.AdamW(groups, lr=cfg.lr)


def make_scheduler(optimizer, cfg: TrainConfig, steps_per_epoch: int):
    """LR theo từng bước: warmup tuyến tính rồi giảm theo cosine về 0."""
    total = cfg.epochs * steps_per_epoch
    warmup = int(cfg.warmup_epochs * steps_per_epoch)

    def lr_lambda(step: int) -> float:
        if warmup > 0 and step < warmup:
            return (step + 1) / warmup
        progress = (step - warmup) / max(1, total - warmup)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def train_one_epoch(model, loader, optimizer, scheduler, criterion, device, grad_clip=0.0):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)                    # 1. forward: (B, 10) logits, CHƯA qua softmax
        loss = criterion(logits, y)          # 2. loss: CrossEntropyLoss = log-softmax + NLL

        optimizer.zero_grad(set_to_none=True)
        loss.backward()                      # 3. backward: tính gradient
        if grad_clip > 0:
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()                     # 4. cập nhật trọng số
        scheduler.step()                     # LR thay đổi theo từng bước

        bs = y.size(0)
        total_loss += loss.item() * bs
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += bs
    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, loader, criterion, device, return_outputs=False):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    all_logits, all_targets = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        total_loss += criterion(logits, y).item() * y.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += y.size(0)
        if return_outputs:
            all_logits.append(logits.float().cpu())
            all_targets.append(y.cpu())
    if return_outputs:
        return total_loss / n, correct / n, torch.cat(all_logits), torch.cat(all_targets)
    return total_loss / n, correct / n


def fit(model: nn.Module, data, cfg: TrainConfig, device, out_dir: str, name: str) -> dict:
    """Huấn luyện, chọn checkpoint tốt nhất theo val accuracy, đánh giá trên test, lưu log.

    Ghi ra out_dir:
        history.csv   - loss/acc train & val, LR, thời gian từng epoch
        best.pt       - trọng số có val accuracy cao nhất
        test_outputs.npz - logits và nhãn trên tập test (để vẽ confusion matrix, tìm mẫu sai)
        summary.json  - tóm tắt số liệu + cấu hình + môi trường
    """
    os.makedirs(out_dir, exist_ok=True)
    model.to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    optimizer = make_optimizer(model, cfg)
    scheduler = make_scheduler(optimizer, cfg, len(data.train))

    n_params = count_params(model)
    print(f"[{name}] params = {n_params:,} | device = {device}")

    history = []
    best_val_acc, best_epoch = -1.0, -1
    ckpt_path = os.path.join(out_dir, "best.pt")
    for epoch in range(1, cfg.epochs + 1):
        lr_now = optimizer.param_groups[0]["lr"]
        with Timer(device) as t:
            tr_loss, tr_acc = train_one_epoch(model, data.train, optimizer, scheduler,
                                              criterion, device, cfg.grad_clip)
        va_loss, va_acc = evaluate(model, data.val, criterion, device)
        history.append({
            "epoch": epoch, "train_loss": tr_loss, "train_acc": tr_acc,
            "val_loss": va_loss, "val_acc": va_acc, "lr": lr_now, "epoch_time": t.elapsed,
        })
        if va_acc > best_val_acc:
            best_val_acc, best_epoch = va_acc, epoch
            torch.save(model.state_dict(), ckpt_path)
        print(f"[{name}] epoch {epoch:2d}/{cfg.epochs} | train loss {tr_loss:.4f} acc {tr_acc:.4f} "
              f"| val loss {va_loss:.4f} acc {va_acc:.4f} | {t.elapsed:.1f}s")

    with open(os.path.join(out_dir, "history.csv"), "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(history[0].keys()))
        writer.writeheader()
        writer.writerows(history)

    # Đánh giá trên test bằng checkpoint tốt nhất theo val (không chọn theo test).
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    te_loss, te_acc, logits, targets = evaluate(model, data.test, criterion, device, return_outputs=True)
    np.savez_compressed(os.path.join(out_dir, "test_outputs.npz"),
                        logits=logits.numpy().astype(np.float32), targets=targets.numpy())

    # Thời gian suy luận trên toàn bộ tập test (sau 1 lượt chạy làm nóng).
    evaluate(model, data.test, criterion, device)
    with Timer(device) as t_inf:
        evaluate(model, data.test, criterion, device)

    epoch_times = [h["epoch_time"] for h in history]
    summary = {
        "name": name,
        "params": n_params,
        "best_epoch": best_epoch,
        "best_val_acc": best_val_acc,
        "test_acc": te_acc,
        "test_loss": te_loss,
        "final_train_acc": history[-1]["train_acc"],
        "final_train_loss": history[-1]["train_loss"],
        "mean_epoch_time": float(np.mean(epoch_times[1:] if len(epoch_times) > 1 else epoch_times)),
        "total_train_time": float(np.sum(epoch_times)),
        "test_inference_time": t_inf.elapsed,
        "data_sizes": data.sizes,
        "norm_mean": data.mean,
        "norm_std": data.std,
        "config": asdict(cfg),
        "env": environment_info(),
    }
    save_json(summary, os.path.join(out_dir, "summary.json"))
    print(f"[{name}] best val acc {best_val_acc:.4f} (epoch {best_epoch}) | test acc {te_acc:.4f}")
    return summary

## Mô hình

- **Softmax:** flatten → `Linear(784, 10)`. Hồi quy logistic đa lớp; ranh giới giữa hai lớp là một siêu phẳng.
- **MLP:** flatten → 784 → 512 → 256 → 10, ReLU + Dropout. Phi tuyến, nhưng vẫn không biết hai pixel nào nằm cạnh nhau.
- **CNN:** hai khối [conv3×3–BN–ReLU]×2 + maxpool (28→14→7), rồi FC 3136 → 128 → 10.
  Conv để `bias=False` vì BatchNorm ngay sau đã có tham số dịch.
  Kích thước được chọn để CNN và MLP có số tham số cùng cỡ.

Cả ba trả về **logits**; softmax nằm trong `CrossEntropyLoss` khi huấn luyện.

In [ ]:
IN_DIM = 28 * 28
NUM_CLASSES = 10


class SoftmaxClassifier(nn.Module):
    """Hồi quy softmax: flatten -> một ánh xạ tuyến tính W x + b -> (softmax trong loss)."""

    def __init__(self, in_dim: int = IN_DIM, num_classes: int = NUM_CLASSES):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(in_dim, num_classes)

    def forward(self, x):               # x: (B, 1, 28, 28)
        return self.fc(self.flatten(x))  # (B, 10)


class MLPClassifier(nn.Module):
    """MLP trên vector flatten: 784 -> 512 -> 256 -> 10, ReLU + Dropout ở các tầng ẩn."""

    def __init__(self, in_dim: int = IN_DIM, hidden=(512, 256), num_classes: int = NUM_CLASSES,
                 dropout: float = 0.2):
        super().__init__()
        layers, prev = [nn.Flatten()], in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(inplace=True), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def conv_bn_relu(c_in: int, c_out: int) -> nn.Sequential:
    # bias=False vì BatchNorm ngay sau đã có tham số dịch (beta), bias của conv là thừa.
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(c_out),
        nn.ReLU(inplace=True),
    )


class CNNClassifier(nn.Module):
    """CNN nhỏ kiểu VGG cho ảnh 28x28:
        [conv-bn-relu x2, maxpool]  1  -> 32 kênh, 28x28 -> 14x14
        [conv-bn-relu x2, maxpool]  32 -> 64 kênh, 14x14 -> 7x7
        flatten (64*7*7 = 3136) -> FC 128 -> dropout -> FC 10
    """

    def __init__(self, in_ch: int = 1, channels=(32, 64), fc_dim: int = 128,
                 num_classes: int = NUM_CLASSES, dropout: float = 0.3):
        super().__init__()
        blocks, prev = [], in_ch
        for c in channels:
            blocks += [conv_bn_relu(prev, c), conv_bn_relu(c, c), nn.MaxPool2d(2)]
            prev = c
        self.features = nn.Sequential(*blocks)
        spatial = 28 // (2 ** len(channels))
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(prev * spatial * spatial, fc_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(fc_dim, num_classes),
        )

    def forward(self, x):
        return self.head(self.features(x))


MODELS = {
    "softmax": SoftmaxClassifier,
    "mlp": MLPClassifier,
    "cnn": CNNClassifier,
}


def build_model(name: str) -> nn.Module:
    return MODELS[name]()

### Số tham số theo từng tầng

In [ ]:
def param_table(model):
    rows = [(name, tuple(p.shape), p.numel()) for name, p in model.named_parameters()]
    return pd.DataFrame(rows, columns=["tham số", "shape", "số phần tử"])


x_demo = torch.randn(2, 1, 28, 28)
for name in E1_NAMES:
    m = build_model(name)
    print(f"{LABEL[name]}: {count_params(m):,} tham số | output {tuple(m(x_demo).shape)}")
    display(param_table(m))

## Huấn luyện

Cả ba mô hình dùng **cùng** seed, split, thứ tự batch, optimizer (AdamW, LR 1e-3, weight decay 1e-4),
cosine schedule và số epoch, để khác biệt kết quả chỉ đến từ kiến trúc.

In [ ]:
for name in E1_NAMES:
    set_seed(SEED)
    data = get_data(DATA_ROOT, batch_size=BATCH_SIZE, loader_seed=SEED)
    model = build_model(name)
    cfg = TrainConfig(epochs=EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY, seed=SEED,
                      extra={"exercise": "E1", "model": name, "batch_size": BATCH_SIZE})
    fit(model, data, cfg, device, os.path.join(RESULTS, name), name=f"E1-{name}")

## Kết quả
Đọc lại các lần chạy đã lưu (chạy được độc lập với ô huấn luyện nếu đã có `results/E1/`).

In [ ]:
runs = {n: load_run(os.path.join(RESULTS, n)) for n in E1_NAMES}
table = pd.DataFrame([summary_row(LABEL[n], runs[n]) for n in E1_NAMES]).set_index("Mô hình")
display(table)
table.to_csv(os.path.join(FIGURES, "e1_table.csv"))

### Đường cong học
Khoảng cách giữa nét chấm (train) và nét liền (validation) cho biết mức overfit.
Để ý val loss của CNN: nó có thể tăng lại trong khi val accuracy vẫn tăng.

In [ ]:
plot_curves(runs, LABEL, COLOR, save_as=os.path.join(FIGURES, "e1_curves.png"))
for n in E1_NAMES:
    h = runs[n]["history"]
    i = int(h["val_loss"].idxmin())
    print(f"{LABEL[n]:8s} val loss thấp nhất {h['val_loss'][i]:.4f} ở epoch {int(h['epoch'][i])}, "
          f"cuối cùng {h['val_loss'].iloc[-1]:.4f} | train loss cuối {h['train_loss'].iloc[-1]:.4f}")

## Phân tích lỗi
### Accuracy theo lớp và confusion matrix

In [ ]:
cms = {n: confusion(runs[n]["targets"], runs[n]["preds"]) for n in E1_NAMES}
per_class = pd.DataFrame({LABEL[n]: (100 * np.diag(cms[n]) / cms[n].sum(1)).round(2) for n in E1_NAMES},
                         index=CLASSES)
display(per_class)

# Confusion matrix của CNN: số trong ô là số ảnh, màu là tỉ lệ theo hàng
cm = cms["cnn"]
norm = cm / cm.sum(1, keepdims=True)
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
for i in range(10):
    for j in range(10):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7,
                    color="white" if norm[i, j] > 0.55 else "black")
ax.set_xticks(range(10), CLASSES, rotation=45, ha="right")
ax.set_yticks(range(10), CLASSES)
ax.set(xlabel="lớp dự đoán", ylabel="lớp thật", title="Confusion matrix — CNN (tập test)")
ax.grid(False)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.savefig(os.path.join(FIGURES, "e1_confusion_cnn.png"))
plt.show()

# Ba mô hình, chỉ phần lỗi (bỏ đường chéo), cùng thang màu
errs = {n: cms[n] / cms[n].sum(1, keepdims=True) * (1 - np.eye(10)) for n in E1_NAMES}
vmax = max(e.max() for e in errs.values())
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, n in zip(axes, E1_NAMES, strict=True):
    im = ax.imshow(errs[n], cmap="Blues", vmin=0, vmax=vmax)
    ax.set_xticks(range(10), SHORT, rotation=60, ha="right", fontsize=7)
    ax.set_yticks(range(10), SHORT, fontsize=7)
    ax.set_title(f"{LABEL[n]} — test acc {100 * runs[n]['summary']['test_acc']:.2f}%")
    ax.grid(False)
fig.colorbar(im, ax=list(axes), fraction=0.02, pad=0.02, label="tỉ lệ lỗi theo hàng")
fig.savefig(os.path.join(FIGURES, "e1_confusion_all.png"))
plt.show()

### Các cặp lớp hay nhầm và mức chồng lấp lỗi giữa ba mô hình

In [ ]:
off = cms["cnn"].copy()
np.fill_diagonal(off, 0)
top = np.dstack(np.unravel_index(np.argsort(off, axis=None)[::-1][:6], off.shape))[0]
pairs = pd.DataFrame([{"Nhãn thật": CLASSES[t], "Bị đoán là": CLASSES[p],
                       **{LABEL[n]: int(cms[n][t, p]) for n in E1_NAMES}} for t, p in top])
display(pairs)

UPPER = [0, 2, 4, 6]  # T-shirt/top, Pullover, Coat, Shirt
for n in E1_NAMES:
    cm_n = cms[n]
    total = cm_n.sum() - np.trace(cm_n)
    inside = sum(cm_n[a, b] for a in UPPER for b in UPPER if a != b)
    print(f"{LABEL[n]:8s} {total:5d} ảnh sai, {100 * inside / total:.1f}% nằm trong nhóm áo thân trên")

wrong = {n: runs[n]["preds"] != runs[n]["targets"] for n in E1_NAMES}
print(f"{100 * (wrong['cnn'] & wrong['mlp']).sum() / wrong['cnn'].sum():.1f}% ảnh CNN sai cũng bị MLP sai")
print(f"{(wrong['cnn'] & wrong['mlp'] & wrong['softmax']).sum()} ảnh cả ba mô hình đều sai")

### Những ảnh CNN đoán sai với độ tự tin cao nhất
Nhiều ảnh trong số này mơ hồ ngay cả với người, có ảnh có vẻ bị gán nhãn sai.

In [ ]:
prob = softmax_np(runs["cnn"]["logits"])
tg, pr = runs["cnn"]["targets"], runs["cnn"]["preds"]
wrong_idx = np.where(pr != tg)[0]
order = wrong_idx[np.argsort(-prob[wrong_idx, pr[wrong_idx]])][:16]
fig, axes = plt.subplots(2, 8, figsize=(13, 4.6))
for ax, i in zip(axes.flat, order, strict=False):
    ax.imshow(TEST_IMAGES[i], cmap="gray_r")
    ax.set_title(f"thật: {CLASSES[tg[i]]}\nđoán: {CLASSES[pr[i]]}\np={prob[i, pr[i]]:.2f}", fontsize=7)
    ax.axis("off")
fig.tight_layout(h_pad=1.5)
fig.savefig(os.path.join(FIGURES, "e1_errors.png"))
plt.show()

## Ghi chú khi viết nhận xét

- So sánh accuracy **và** số tham số: CNN ít tham số hơn MLP nhưng chính xác hơn — vì sao (tính cục bộ, chia sẻ trọng số)?
- Đường cong: mô hình nào underfit, mô hình nào overfit; val loss tăng trong khi val accuracy vẫn tăng nghĩa là gì?
- Lỗi tập trung ở lớp nào, cặp lớp nào; các ảnh sai tự tin cho biết gì về dữ liệu.

## Lưu kết quả

In [ ]:
# Nén kết quả để tải về (Colab). Chạy trên máy cá nhân thì kết quả đã nằm sẵn trong thư mục results/ và figures/.
import shutil

archive = shutil.make_archive(f"{EXERCISE}_results", "zip", ".", base_dir="results")
shutil.make_archive(f"{EXERCISE}_figures", "zip", ".", base_dir="figures")
print("Đã tạo", archive, "và", f"{EXERCISE}_figures.zip")
try:
    from google.colab import files
    files.download(archive)
    files.download(f"{EXERCISE}_figures.zip")
except ImportError:
    pass